# Simulación Monte Carlo: Europea, Asiática y Lookback

Modelo: movimiento browniano geométrico

$$S_{t+1} = S_t \cdot e^{(r_f-\sigma^2/2)\Delta t + \sigma \sqrt{\Delta t}\, X}, \quad X\sim N(0,1)$$

Idea (como en los apuntes): se simulan **M trayectorias** con **n pasos** y se guardan en una matriz. Las tres opciones usan **las mismas trayectorias**; solo cambia el payoff.

## 1. Parámetros

In [1]:
import numpy as np

t_0 = 4        # instante inicial (años)
T   = 5        # vencimiento (años)
S_0 = 100      # precio inicial
K   = 80       # strike
rf  = 0.1      # tipo libre de riesgo
vol = 0.3      # volatilidad (sigma)
M   = 100000   # nº de trayectorias
n   = 12       # nº de pasos (ej. mensual si T - t_0 = 1 año)

delta_t = (T - t_0) / n   # tamaño de cada paso

## 2. Crear la matriz de trayectorias

- Columna 0 → $S_0$
- Columnas 1..n → los factores $e_1, e_2, \dots, e_n$ (cada uno con su propio número aleatorio $X$)
- `np.cumprod(axis=1)` multiplica en horizontal: $S_1 = S_0 e_1$, $S_2 = S_0 e_1 e_2$, ...

In [2]:
rng = np.random.default_rng(seed=0)
X = rng.standard_normal((M, n))                         # aleatorios: M filas (trayectorias) x n pasos

e = np.exp((rf - vol**2 / 2) * delta_t + vol * np.sqrt(delta_t) * X)   # factores e_1 ... e_n

S = np.empty((M, n + 1))
S[:, 0]  = S_0
S[:, 1:] = e

S = np.cumprod(S, axis=1)       # cada fila es una trayectoria completa
S[:3]                           # vemos las 3 primeras trayectorias

array([[100.        , 101.55921992, 100.8651819 , 107.1072163 ,
        108.58120261, 104.1353306 , 107.94153011, 121.40144029,
        132.38385477, 125.1288201 , 112.65593905, 107.22662317,
        108.10541721],
       [100.        ,  82.13793332,  80.96649015,  73.01899753,
         68.84700801,  65.97895915,  64.491074  ,  67.13855104,
         73.81973071,  73.33793125,  82.93029999,  78.64753445,
         81.45097284],
       [100.        , 108.63529256, 110.02651359, 103.63921475,
         96.1274921 ,  92.81594695,  95.03747216,  87.48077819,
         86.30497883,  85.51410487,  90.02642753,  92.13700233,
         95.453206  ]])

## 3. Payoffs

Se calcula el payoff de cada trayectoria (un vector de M valores), se hace la media (`np.mean`) y se descuenta:

$$C_0 = e^{-r_f (T-t_0)} \cdot \text{media(payoff)}$$

In [3]:
descuento = np.exp(-rf * (T - t_0))

### Europea
Solo importa el precio final: $\max(S_T - K, 0)$

In [4]:
payoff_europea = np.maximum(S[:, -1] - K, 0)
precio_europea = descuento * np.mean(payoff_europea)

### Asiática
Usa la **media** del precio a lo largo de la trayectoria: $\max(\bar{S} - K, 0)$

In [5]:
media_S = np.mean(S[:, 1:], axis=1)          # media de cada fila (sin contar S_0)
payoff_asiatica = np.maximum(media_S - K, 0)
precio_asiatica = descuento * np.mean(payoff_asiatica)

### Lookback
Usa el **máximo** de la trayectoria: $\max(S_{max} - K, 0)$

In [6]:
max_S = np.max(S, axis=1)                   # máximo de cada fila
payoff_lookback = np.maximum(max_S - K, 0)
precio_lookback = descuento * np.mean(payoff_lookback)

## 4. Resultados

In [7]:
print(f"Call europea : {precio_europea}")
print(f"Call asiática: {precio_asiatica}")
print(f"Call lookback: {precio_lookback}")

Call europea : 29.47339336005617
Call asiática: 23.622474042365198
Call lookback: 42.96842695941744
